# Module 09 - 실습 2 솔루션: TokenBucket Rate Limiter

## 학습 목표
- 토큰 버킷 알고리즘의 개념을 이해한다
- `TokenBucket` 클래스를 구현할 수 있다
- Rate Limiter로 API 호출 속도를 제어할 수 있다

In [ ]:
import sys
sys.path.insert(0, '../..')

import asyncio
import time

print("Rate Limiter 솔루션 준비 완료")

## 실습 1 솔루션: TokenBucket 클래스 구현

In [ ]:
class TokenBucket:
    """토큰 버킷 기반 Rate Limiter.
    
    초당 허용 요청 수를 제한합니다.
    
    Args:
        rate: 초당 토큰 보충 속도 (예: 5 = 초당 5개)
        capacity: 버킷 최대 용량
    """
    
    def __init__(self, rate: float, capacity: int):
        self.rate = rate
        self.capacity = capacity
        self.tokens = capacity          # 처음에는 가득 참
        self.last_refill = time.monotonic()
        self._lock = asyncio.Lock()
    
    async def acquire(self) -> None:
        """토큰을 1개 획득합니다. 없으면 대기합니다."""
        async with self._lock:
            await self._refill()
            
            while self.tokens < 1:
                # 토큰이 보충될 때까지 대기
                wait_time = (1 - self.tokens) / self.rate
                await asyncio.sleep(wait_time)
                await self._refill()
            
            self.tokens -= 1
    
    async def _refill(self) -> None:
        """경과 시간에 따라 토큰을 보충합니다."""
        now = time.monotonic()
        elapsed = now - self.last_refill
        self.tokens = min(
            self.capacity,
            self.tokens + elapsed * self.rate,
        )
        self.last_refill = now

In [ ]:
# 검증 셀
bucket = TokenBucket(rate=10, capacity=5)
assert hasattr(bucket, 'rate'), "rate 속성이 없습니다"
assert hasattr(bucket, 'capacity'), "capacity 속성이 없습니다"
assert hasattr(bucket, 'tokens'), "tokens 속성이 없습니다"
assert bucket.tokens == 5, f"초기 토큰이 capacity(5)여야 합니다. 현재: {bucket.tokens}"
print("✅ 실습 1-1 완료! TokenBucket 초기화 성공.")

## 실습 2 솔루션: acquire() 테스트

In [ ]:
limiter = TokenBucket(rate=5, capacity=5)
call_times = []

start = time.time()
for i in range(10):
    await limiter.acquire()  # 토큰 획득 (없으면 대기)
    call_times.append(time.time() - start)
    print(f"호출 {i+1}: {call_times[-1]:.2f}초")

total_time = time.time() - start
print(f"\n10개 API 호출 완료: {total_time:.2f}초")

In [ ]:
# 검증 셀
assert len(call_times) == 10, f"10번 호출해야 합니다. 현재: {len(call_times)}"
assert total_time >= 0.8, f"rate limiter가 동작하지 않습니다. 소요 시간: {total_time:.2f}초"
print(f"✅ 실습 2 완료! Rate Limiter가 올바르게 동작합니다. (총 {total_time:.2f}초)")

## 실습 3 솔루션: safe_api_call 함수

In [ ]:
api_rate_limiter = TokenBucket(rate=3, capacity=3)  # 초당 3개 허용

async def safe_api_call(url: str) -> str:
    """Rate Limit을 지키며 API를 호출합니다."""
    await api_rate_limiter.acquire()  # 토큰 획득 (없으면 대기)
    # 실제 API 호출 시뮬레이션
    await asyncio.sleep(0.05)
    return f"OK: {url}"

urls = [f"https://api.example.com/data/{i}" for i in range(5)]
start = time.time()
responses = await asyncio.gather(*[safe_api_call(url) for url in urls])
elapsed = time.time() - start

print(f"5개 API 호출 완료: {elapsed:.2f}초")
for r in responses:
    print(f"  {r}")

In [ ]:
# 검증 셀
assert len(responses) == 5, f"응답이 5개여야 합니다. 현재: {len(responses)}"
assert all(r is not None for r in responses), "모든 응답이 None이 아니어야 합니다"
assert elapsed >= 0.5, f"Rate limiter가 동작하지 않습니다. 소요 시간: {elapsed:.2f}초"
print(f"✅ 실습 3 완료! safe_api_call이 Rate Limit을 지킵니다. (총 {elapsed:.2f}초)")

## 정리

이번 실습에서 배운 내용:
1. 토큰 버킷 알고리즘으로 API 호출 속도를 사전에 제한할 수 있다
2. `rate`(초당 보충)와 `capacity`(최대 용량)로 Rate Limit을 조절한다
3. `asyncio.Lock()`으로 동시성 문제를 방지한다